In [ ]:
!pip install --upgrade pip
!pip install huggingface_hub
!pip install pandas torch
!pip install tdqm
!pip install torch
!pip install --upgrade "transformers==4.53.0"
!pip install accelerate
#!pip install -U jupyter ipywidgets


In [ ]:
from huggingface_hub import login

# 🔐 ใส่ token ของคุณตรงนี้
login(token="*********************************")

In [ ]:
import pandas as pd
merged_table=pd.read_csv('../1.ML/app/ml.csv') 

In [ ]:
import os
os.environ["OMP_NUM_THREADS"] = "32"
os.environ["MKL_NUM_THREADS"] = "32"

import torch
torch.set_num_threads(32)

from transformers import AutoModelForCausalLM, AutoTokenizer

# ตั้งค่าชื่อโมเดลทั้งหมดที่คุณต้องการทดสอบ
model_names = {
    "Qwen2.5-3B-Instruct": "Qwen/Qwen2.5-3B-Instruct",
    "meta-llama/Llama-3.2-3B-Instruct": "meta-llama/Llama-3.2-3B-Instruct",
    "Phi-4-mini-instruct": "microsoft/Phi-4-mini-instruct",
}

# โหลดโมเดลและ tokenizer
models = {}
tokenizers = {}

for key, name in model_names.items():
    print(f"🔄 Loading {key}...")
    tokenizer = AutoTokenizer.from_pretrained(name, trust_remote_code=True)

    model = AutoModelForCausalLM.from_pretrained(
        name,
        torch_dtype=torch.float16 if torch.cuda.is_available() else "auto",
        device_map="auto",  # ใช้ GPU 'cuda:0'
        trust_remote_code=True
    ).eval()

    tokenizers[key] = tokenizer
    models[key] = model


In [ ]:
import re
from json import loads


batch_size =2  # หรือขนาดที่ต้องการ

# Define start and stop indices
start = 0
stop = len(merged_table)

# Slice the data according to start/stop
prompts = merged_table['text'][start:].tolist()
agents = merged_table['ml_description'][start:].tolist() 



for model_key in models:

    tokenizer = tokenizers[model_key]
    model = models[model_key]
    
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    
    # สำคัญ! ตั้ง padding_side เป็น left สำหรับ decoder-only models
    tokenizer.padding_side = 'left'
    
    # แบ่ง batch และ process
    for i in range(0, stop, batch_size):
        #merged_table.to_csv("OpensourceModel.csv", index=False)
        print(start + i) 
        batch_end = min(i + batch_size, len(prompts))
        batch_prompts = prompts[i:batch_end]
        batch_agents = agents[i:batch_end]

        formatted_prompts = [tokenizer.apply_chat_template([
                        {"role": "system", "content": """
        You are a medical coding specialist. Analyze discharge summaries and predict ICD-10 codes.

        INSTRUCTIONS:
        1. Return relevant codes
        2. Use standard ICD-10 format (e.g., I50.9, N18.6)
        3. Provide concise descriptions
        4. Include probability scores (0.0-1.0)

        OUTPUT FORMAT (JSON):
        {
        "E11.9": "Type 2 diabetes without complications (0.95)",
        "I25.9": "Chronic ischemic heart disease (0.87)",
        "N18.6": "End stage renal disease (0.82)"
        }
        """},
                    {"role": "user", "content": f"Discharge Summary:\n{prompt}\n\nPredict ICD-10 codes:"}
                    ], 
                    tokenize=False,
                    add_generation_prompt=True) 
                    for prompt in batch_prompts
                    ]

        
        # Tokenize with left padding
        batch_inputs = tokenizer(
            formatted_prompts,
            return_tensors="pt",
            padding=True,
            truncation=True
        )


        # Move to device
        batch_inputs = {k: v.to(model.device) for k, v in batch_inputs.items()}


        sequences = model.generate(
            **batch_inputs,
            max_new_tokens=1024,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
            temperature=1.0,
            top_p=1,
            top_k=50
        )
    
        # Process results - FIXED INDEX CALCULATION
        for j, seq in enumerate(sequences):

            input_ids = batch_inputs["input_ids"][j]
            generated_text = tokenizer.decode(seq[len(input_ids):], skip_special_tokens=True)
            
            # Extract JSON
            match = re.search(r'\{.*\}', generated_text, re.DOTALL)
            if match:
                try:
                    json_str = match.group(0)
                    keys = list(loads(json_str).keys())
           
                except:
                    keys = []
                    json_str = "INVALID_JSON"
            else:
                json_str = "NO_JSON_FOUND"
                keys = []
            
            # FIXED: Calculate correct row index in original dataframe
            actual_row_idx = start + i + j  # This is the actual index in merged_table
            
            row_idx = merged_table.index[actual_row_idx]  # Get the pandas index

            merged_table.at[row_idx, f"{model_key}_description"] = json_str 
            merged_table.at[row_idx, f"{model_key}"] = repr(keys)

In [ ]:
merged_table.to_csv("OpensourceModel.csv", index=False)